# masrafAI — açıklama toplu üretimi (Kaggle + vLLM + Qwen3-8B)

**Dataset hazırlığı.** İki şey gerekli; aynı ya da AYRI dataset olabilir,
Hücre 3 ikisini de `/kaggle/input` içinde nerede olursa bulur:

!!faz_b_aciklama_uretimi klasörünün bir kopyasını ve ortak klasörünün bir kopyasını batch olarak hazırladığınız açıklama klasörü ile birlikte yeni bir dosyaya zipleyin.
!!Daha sonra bu zipi kaggle notebookta file -> input -> load dataset diyerek ekleyebilirsiniz.

bu dosyalarda kullanılacak önemli modüller:
1. **Kod:** `faz_b_aciklama_uretimi/` (içinde `__init__.py`, `aciklama_uretim_core.py`,
   `aciklama_toplu_uret.py`, `aciklama_analiz.py`) ve `ortak/` (`__init__.py`,
   `cift_grup.py`). Alt klasör yapısı KORUNMALI; flat `.py` yükleme çalışmaz.
2. **Batch:** `batch_NNNN.json` dosyaları + `durum.json` (`batch_hazirla.py` çıktısı).

Kaç batch / kaç kayıt olduğu `durum.json`'dan okunur; hücrelerde elle sayı
güncellemen GEREKMEZ. Kod dataset'ini güncellediyseysen notebook'taki eski
sürümü Input panelinden çıkar (Hücre 3 birden çok kaynak görürse uyarır).

Kaggle ayarları: Internet **açık**, Accelerator **GPU T4 ×2**.
Gözetimsiz koşu: **Save Version → Save & Run All (commit)**.
Tempo ~66 kayıt/dk (Qwen3-8B fp16, 16 worker); 25k ≈ 6,5 saat.


## Hücre 1 — kurulum

In [ ]:
#Hücrenin sol üstündeli çalıştır butonunun. bitmesini bekleyin. Çalıştırma tamamlandıktan sonra aşağıdaki hücreyi çalıştırın. Hep uygulayın bunu.
!pip install -q vllm hf_transfer
!nvidia-smi --query-gpu=name,memory.total --format=csv


## Hücre 2 — ortam (HF_TOKEN)

In [ ]:
# Hugging Face hesabınızdan profilinize tıklayıp acces token sekmesinden token alıp kaggle notebookta secrets kısmına HF_TOKEN olarak ekleyin. 
# Bu aşama olmadan da açıklama üretilir ama modelin hızlı inmesini sağlayan bir aşama bu.
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN ayarlandi")
except Exception as e:
    print("HF_TOKEN yok, anonim indirme:", e)

## Hücre 3 — dosyaları kopyala + doğrula

Kod (`faz_b_aciklama_uretimi/` + `ortak/`) ve batch (`batch_NNNN.json` + `durum.json`)
`/kaggle/input`'ta nerede olursa bulunur; ayrı dataset olabilirler.
Beklenen sayılar `durum.json`'dan okunur (sabit DEĞİL): çıktıda `batch: N / N`
eşleşmeli ve `yeni kurallar: True` olmalı.

**ÖN KOŞUL:** dataset flat `.py` değil, `faz_b_aciklama_uretimi/` + `ortak/` alt klasörlerini
(her ikisinde `__init__.py` ile) KORUYARAK paketlenmiş olmalı. Eski flat
dataset ya da eski `core` bu hücreyi kırar. Birden çok kod kaynağı veya birden
çok `durum.json` bağlıysa hücre durur: fazlasını Input panelinden çıkar.


In [ ]:
##
import shutil, glob, os, json

# 1) KOD kaynagi: faz_b_aciklama_uretimi/ + ortak/ iceren dataset (nerede olursa)
kod = glob.glob("/kaggle/input/**/faz_b_aciklama_uretimi/aciklama_uretim_core.py", recursive=True)
assert kod, ("faz_b_aciklama_uretimi/aciklama_uretim_core.py bulunamadi -- kod dataset'i ekli mi, "
             "faz_b_aciklama_uretimi/ alt klasoruyle mi paketlendi? -> !find /kaggle/input -maxdepth 6 -name '*.py'")
KOD_KOKU = sorted({os.path.dirname(os.path.dirname(p)) for p in kod})
assert len(KOD_KOKU) == 1, f"BIRDEN COK kod kaynagi bagli: {KOD_KOKU} -- eskisini Input panelinden cikar"
KOD_KOKU = KOD_KOKU[0]

# 2) BATCH kaynagi: durum.json + batch_NNNN.json (ayni ya da ayri dataset)
durumlar = glob.glob("/kaggle/input/**/durum.json", recursive=True)
assert durumlar, "durum.json bulunamadi -- batch dataset'i ekli mi?"
assert len(durumlar) == 1, f"BIRDEN COK durum.json bagli: {durumlar} -- fazlasini cikar"
BATCH_KOKU = os.path.dirname(durumlar[0])

HEDEF = "/kaggle/working/aciklama_25k"
os.makedirs(HEDEF, exist_ok=True)
for f in glob.glob(f"{BATCH_KOKU}/batch_[0-9][0-9][0-9][0-9].json") + [durumlar[0]]:
    shutil.copy(f, HEDEF)

# 3) faz_b_aciklama_uretimi/ + ortak/ alt klasor yapisi KORUNARAK /kaggle/working'e
#    ('python -m faz_b_aciklama_uretimi.X' import'lari repo kokunden = /kaggle/working calisir)
for alt in ("faz_b_aciklama_uretimi", "ortak"):
    kaynak_alt = os.path.join(KOD_KOKU, alt)
    assert os.path.isdir(kaynak_alt), f"{alt}/ kod dataset'te yok -- paketleme eksik"
    shutil.copytree(kaynak_alt, f"/kaggle/working/{alt}", dirs_exist_ok=True)
    open(f"/kaggle/working/{alt}/__init__.py", "a").close()

# 4) beklenenler durum.json'dan (sabit sayi YOK)
durum = json.load(open(f"{HEDEF}/durum.json", encoding="utf-8"))
BEKLENEN_BATCH = durum["batch_sayisi"]
BEKLENEN_KAYIT = durum["toplam_secilen"]
BATCH_SIZE = durum.get("config", {}).get("batch_size", "?")

b = sorted(glob.glob(f"{HEDEF}/batch_[0-9][0-9][0-9][0-9].json"))
c = glob.glob(f"{HEDEF}/*_ciktilar.json")
core = open("/kaggle/working/faz_b_aciklama_uretimi/aciklama_uretim_core.py", encoding="utf-8").read()
kurallar_tamam = all(k in core for k in ("vurgu_fazlasi", "tema_halusinasyonu", "_KATEGORI_GENEL_AD"))

print("kod koku  :", KOD_KOKU)
print("batch koku:", BATCH_KOKU)
print(f"batch     : {len(b)} / {BEKLENEN_BATCH}   (hedef {BEKLENEN_KAYIT} kayit, batch_size {BATCH_SIZE})")
print("durum     :", os.path.exists(f"{HEDEF}/durum.json"))
print("faz_b_aciklama_uretimi     :", sorted(os.listdir("/kaggle/working/faz_b_aciklama_uretimi")))
print("ortak     :", sorted(os.listdir("/kaggle/working/ortak")))
print("ciktilar  :", len(c), c)
print("tarih     :", json.load(open(b[0]))[0].get("fatura_tarihi"))
print("core      :", len(core), "byte |", "yeni kurallar:", kurallar_tamam)
assert len(b) == BEKLENEN_BATCH, f"batch dosyasi {len(b)}, durum.json {BEKLENEN_BATCH} diyor -- zip eksik"
assert kurallar_tamam, "ESKI core kopyalanmis -- eski kod dataset'ini cikar"


## Hücre 4 — vLLM sunucusu

Model indirmesi 10-20 dk sürer ve bu sırada hücre **hiçbir şey basmaz** (log `vllm.log`'a gider) —
donmuş değildir. Hata alırsan: `!tail -50 /kaggle/working/vllm.log`

In [ ]:
# Modelin indirilmesi
import subprocess, time, requests

MODEL = "Qwen/Qwen3-8B"          # yedek: "Qwen/Qwen3-4B-Instruct-2507"
log = open("/kaggle/working/vllm.log", "w")
cmd = ["python", "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL,
       "--dtype", "half",
       "--tensor-parallel-size", "2",
       "--max-model-len", "2048",
       "--gpu-memory-utilization", "0.92",
       "--enable-prefix-caching",
       "--generation-config", "vllm",
       "--enforce-eager",
       "--port", "8000"]
sunucu = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

url = "http://localhost:8000/v1/models"
hazir = False
for i in range(360):                     # 60 dk
    if sunucu.poll() is not None:
        print("SUNUCU OLDU, exit =", sunucu.returncode)
        break
    try:
        if MODEL in str(requests.get(url, timeout=5).json()):
            print("HAZIR, gecen sure (sn) =", i * 10)
            hazir = True
            break
    except Exception:
        pass
    time.sleep(10)
else:
    print("60 dk doldu - !tail -50 /kaggle/working/vllm.log")

# /v1/models 200 donse de uretim calismiyor olabilir: gercek istek at.
if hazir:
    r = requests.post("http://localhost:8000/v1/chat/completions",
                      json={"model": MODEL, "max_tokens": 16,
                            "messages": [{"role": "user", "content": "Merhaba de."}]},
                      timeout=180)
    print("duman testi:", r.status_code, r.json()["choices"][0]["message"]["content"][:80])

## Hücre 5 — tam üretim

`durum.json`'daki **tüm** batch'ler işlenir (runner kendi okur); kaç batch
olduğunu elle yazman gerekmez. Kesilirse **aynı hücreyi tekrar çalıştır** --
`*_ciktilar.json` dosyalarından kaldığı yerden devam eder.

Log dosyaya gidiyor: runner fatura başına bir satır basıyor (notebook çıktısına
gömülmesin), `[ilerleme]` satırları canlı görünür. `--ilerleme` batch boyutundan
küçük olmalı yoksa hiç ilerleme basmaz.


In [ ]:
%%time
# Tek makine: --batch YOK -> runner durum.json'daki TUM batch'leri isler,
# kaç batch oldugunu elle yazmana gerek yok. Cok makineli bolme yapacaksan
# her makineye AYRIK aralik ver: makine A --batch 1-13 / makine B --batch 14-25
# NOT: 2>&1 YOK -> fatura satirlari log'a, [ilerleme] satirlari hucrede canli.
# --ilerleme batch_size'dan KUCUK olmali (Hucre 3 batch_size'i basiyor).
!cd /kaggle/working && python -u -m faz_b_aciklama_uretimi.aciklama_toplu_uret \
    --cikti-dizini /kaggle/working/aciklama_25k \
    --saglayici vllm --host http://localhost:8000/v1 \
    --model Qwen/Qwen3-8B \
    --sicaklik-tavani 0.9 --cooldown-min 0 --workers 16 \
    --ilerleme 200 > /kaggle/working/uretim.log
!tail -60 /kaggle/working/uretim.log


## Hücre 6 — özet + indirilecek paket

try/except ile sarılı: üretim yarım kalsa bile bu hücre patlamaz, commit yeşil biter
ve `/kaggle/working` versiyon çıktısı olarak kaydedilir.

In [ ]:
import json, glob, collections, zipfile, os
try:
    durum = json.load(open("/kaggle/working/aciklama_25k/durum.json", encoding="utf-8"))
    hedef_batch, hedef_kayit = durum["batch_sayisi"], durum["toplam_secilen"]
    dosyalar = sorted(glob.glob("/kaggle/working/aciklama_25k/*_ciktilar.json"))
    toplam, ih, kat = 0, collections.Counter(), collections.Counter()
    for f in dosyalar:
        C = json.load(open(f))
        toplam += len(C)
        for v in C.values():
            kat[v["aciklama_kategorisi"]] += 1
            for x in (v.get("kalan_ihlaller") or []):
                ih[x] += 1
    print(f"cikti dosyasi: {len(dosyalar)} / {hedef_batch}   uretilen kayit: {toplam} / {hedef_kayit}")
    print("kategori:", dict(kat))
    print("kalan ihlaller:", dict(ih.most_common(12)))

    with zipfile.ZipFile("/kaggle/working/ciktilar.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in dosyalar:
            z.write(f, os.path.basename(f))
        if os.path.exists("/kaggle/working/uretim.log"):
            z.write("/kaggle/working/uretim.log", "uretim.log")
    print("zip:", os.path.getsize("/kaggle/working/ciktilar.zip") // 1024, "KB -> /kaggle/working/ciktilar.zip")
except Exception as e:
    print("ozet basarisiz (uretim yarim kalmis olabilir):", type(e).__name__, e)

!cd /kaggle/working && python -m faz_b_aciklama_uretimi.aciklama_analiz --cikti-dizini /kaggle/working/aciklama_25k || true
# Burada sağ paneldeki kaggle/working/cikti.zip dosyası açıklamalarınızı içerir. Onu indirmeniz yeterli
# zipi açıp içindeki batch dosyalarını klaösrün içinden alıp $D/aciklama içerisine koyun.
# ve nasıl-calisir.md dosyasında açıklama birleştir adımı ile devam edin.